# Capstone: Split effect, GRU

The professor's GRU under both splits (by time, and at random as Gad et al. split), on REFIT,
UK-DALE and CASAS: do networks look better than trees only because of the split?

**Settings:** Accelerator = **GPU**, Internet = **On** (CASAS download and notifications), dataset
`capstone-bundle` (latest version) under Input. Save Version > Save & Run All (Commit).

**A new notebook copies none of these settings.**

In [ ]:
import glob, os, shutil, subprocess
# os.environ['NTFY_URL'] = 'https://ntfy.sh/<your-topic>'   # used only if the NTFY_URL secret is missing
found = glob.glob('/kaggle/input/**/habits', recursive=True)
assert found, 'No bundle: add the capstone-bundle dataset with Add Input (right-hand panel), then rerun'
bundle = os.path.dirname(found[0])
shutil.copytree(bundle, '/kaggle/working/capstone', dirs_exist_ok=True)
os.chdir('/kaggle/working/capstone')
os.makedirs('logs', exist_ok=True)
print('bundle from', bundle)
!pip install -q paho-mqtt
import torch, sklearn, pandas, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| sklearn', sklearn.__version__,
      '| pandas', pandas.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (check the Accelerator setting)')
import lightgbm, xgboost, catboost
print('lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| catboost', catboost.__version__)
sys.path.insert(0, 'scripts')
from kaggle_steps import ADDRESS, notify, run_step
print('ntfy:', 'on' if ADDRESS else 'off (no NTFY_URL secret or first-line topic)')
notify('Split effect, GRU: started', 'CASAS download, then the runs', 'default', 'rocket')

In [ ]:
!mkdir -p /kaggle/temp && wget -q -c -O /kaggle/temp/casas.zip "https://zenodo.org/records/15708568/files/labeled_data.zip?download=1"
!ls -l /kaggle/temp/casas.zip   # should be 236037656 bytes
ok = run_step('CASAS prepare', 'CAPSTONE_DATASET=casas python -u scripts/prepare_casas.py /kaggle/temp/casas.zip',
              'logs/casas_prepare.log', summary_lines=3)
casas_ok = ok  # without CASAS, its steps are skipped (REFIT and UK-DALE still run)

In [ ]:
DATASETS = ['refit', 'ukdale', 'casas']   # to rerun only CASAS: ['casas']
ok &= run_step('Tests', 'python -m pytest -q tests/test_gad.py tests/test_habit_networks.py', 'logs/tests.log', summary_lines=2)
for ds in DATASETS:
    if ds != 'casas' or casas_ok:
        ok &= run_step(f'{ds} GRU, both splits', 'python -u scripts/eval_split_effect.py --models GRU',
                       f'logs/{ds}_split_gru.log', env={'CAPSTONE_DATASET': ds})

In [ ]:
import zipfile, pathlib
with zipfile.ZipFile('/kaggle/working/split_networks.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in [*pathlib.Path('data/results').rglob('*'), *pathlib.Path('logs').rglob('*')]:
        if f.is_file() and (f.suffix == '.log' or f.name.startswith(('split_effect', 'gad_')) or 'casas' in f.parts):
            zf.write(f, str(f))
print(round(pathlib.Path('/kaggle/working/split_networks.zip').stat().st_size / 1e6, 2), 'MB')
shutil.rmtree('/kaggle/working/capstone', ignore_errors=True)  # keep the Output to the zip
notify('Split effect, GRU: FINISHED' if ok else 'Split effect, GRU: finished WITH ERRORS',
       'split_networks.zip is in the Output tab', 'default' if ok else 'high', 'tada' if ok else 'warning')